In [3]:
from Quantum_dot import *
from random_PS import *
from random import seed
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
from qiskit.quantum_info import Statevector
from entropy import RDM_entropy

exact_model = Quantum_dot(N=10, Omega=1.0)
analog_QD = analog_QD(exact_model, J=0.01 ,theta=0.01 ,DeltaE=1)

def state_rank(rank,seed = None):
    #return a random state with the given rank
    if seed is not None:
        seed(seed)
    state = np.zeros(2**exact_model.N, dtype='complex128')
    for i in range(rank):
        rand_state = rand_product_state(exact_model.N)
        state += rand_state
    return state / np.linalg.norm(state)

def long_time_evolve(initial_state, num_steps=15):
    #return the long-time analog simulation error
    exact_state = initial_state
    analog_state = initial_state
    for i in range(num_steps):
        exact_state = exact_model.one_step_evolve(exact_state)
        for j in range(10):
            U, perm = analog_QD.time_dependent_term((i+0.1*j)*time_step)
            analog_state = U @ analog_state
    return np.linalg.norm(exact_state - analog_state)

In [5]:
seed(123456)
entropy_list = []
error_list = []
for rank in tqdm([2,2,2,2,3,3,4,5,6,7,8,9,10]):
    initial_state = state_rank(rank)
    entropy_list.append(RDM_entropy(initial_state, 2)[-1])
    error_list.append(long_time_evolve(initial_state))    

  0%|          | 0/13 [00:43<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
plt.plot(entropy_list,error_list,color='0.3', marker='o', markersize=5, mfc='#e19d92', mec='k', linestyle='dashed', markeredgewidth=0.5,label='t=1.5')
plt.xlabel('Entanglement Entropy')
plt.ylabel('Analog Simulation Error')
plt.title('1D QIMF')
plt.legend()
plt.show()